# Part 2: Transform, Clean, and Analyze
## AI Labor Markets - ELT Pipeline
### AI 620: Data Engineering for AI Systems

This notebook performs data quality assessment, transformation, cleaning, and exploratory data analysis on the three datasets extracted by our ELT pipeline:
1. **HackerNews Stories** - AI/ML job discussions from the HN API
2. **AI Jobs Dataset** - Salary and job market data from Kaggle
3. **Stock Data** - AI company stock prices from Yahoo Finance

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

from config import PROCESSED_DIR, CLEANED_DIR, VIZ_DIR
from load_data import load_csv, save_cleaned
from transform import (
    assess_quality, clean_hn_data, clean_jobs_data,
    clean_stock_data, generate_summary_statistics,
)

print('All imports successful.')

## 1. Load Processed Data

In [ ]:
hn_df = load_csv(PROCESSED_DIR, 'hn_stories.csv')
jobs_df = load_csv(PROCESSED_DIR, 'ai_jobs_dataset.csv')
stock_df = load_csv(PROCESSED_DIR, 'stock_data.csv')

print(f'HackerNews Stories: {hn_df.shape}')
print(f'AI Jobs Dataset:    {jobs_df.shape}')
print(f'Stock Data:         {stock_df.shape}')

## 2. Data Quality Assessment

We will now check for missing values, duplicates, and data types across all three datasets.

In [ ]:
hn_quality = assess_quality(hn_df, 'HackerNews Stories')

In [ ]:
jobs_quality = assess_quality(jobs_df, 'AI Jobs Dataset')

In [ ]:
stock_quality = assess_quality(stock_df, 'Stock Data')

In [ ]:
quality_summary = pd.DataFrame({
    'Dataset': ['HackerNews', 'AI Jobs', 'Stock Data'],
    'Rows': [hn_quality['shape'][0], jobs_quality['shape'][0], stock_quality['shape'][0]],
    'Columns': [hn_quality['shape'][1], jobs_quality['shape'][1], stock_quality['shape'][1]],
    'Missing Values': [hn_quality['missing_total'], jobs_quality['missing_total'], stock_quality['missing_total']],
    'Duplicates': [hn_quality['duplicate_count'], jobs_quality['duplicate_count'], stock_quality['duplicate_count']],
})
print('\nData Quality Summary:')
print(quality_summary.to_string(index=False))

## 3. Transformation and Cleaning

In this section we clean all three datasets by removing duplicates, handling missing values, standardizing dates, and creating new features.

In [ ]:
hn_clean = clean_hn_data(hn_df.copy())
hn_clean.head()

In [ ]:
jobs_clean = clean_jobs_data(jobs_df.copy())
jobs_clean.head()

In [ ]:
stock_clean = clean_stock_data(stock_df.copy())
stock_clean.head()

In [ ]:
generate_summary_statistics(jobs_clean, 'AI Jobs (Cleaned)')

In [ ]:
generate_summary_statistics(stock_clean, 'Stock Data (Cleaned)')

In [ ]:
save_cleaned(hn_clean, 'hn_stories_cleaned')
save_cleaned(jobs_clean, 'ai_jobs_cleaned')
save_cleaned(stock_clean, 'stock_data_cleaned')

## 4. Exploratory Analysis and Visualization

### 4a. Temporal Analysis: AI Company Stock Prices Over Time

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10), gridspec_kw={'height_ratios': [3, 1]})

stock_clean['Date'] = pd.to_datetime(stock_clean['Date'])
for ticker in stock_clean['Ticker'].unique():
    ticker_data = stock_clean[stock_clean['Ticker'] == ticker]
    axes[0].plot(ticker_data['Date'], ticker_data['Close'], label=ticker, linewidth=1.5)

axes[0].set_title('AI/Tech Company Stock Prices (2020-2025)', fontsize=16, fontweight='bold')
axes[0].set_ylabel('Closing Price (USD)', fontsize=12)
axes[0].legend(fontsize=11, loc='upper left')
axes[0].grid(True, alpha=0.3)

monthly_vol = stock_clean.groupby(['YearMonth', 'Ticker'])['Volume'].mean().reset_index()
for ticker in stock_clean['Ticker'].unique():
    t_data = monthly_vol[monthly_vol['Ticker'] == ticker]
    axes[1].plot(range(len(t_data)), t_data['Volume'] / 1e6, label=ticker, alpha=0.7)

axes[1].set_title('Monthly Average Trading Volume', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Volume (Millions)', fontsize=11)
axes[1].set_xlabel('Time (Monthly)', fontsize=11)
axes[1].legend(fontsize=9, loc='upper right')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(VIZ_DIR, 'temporal_stock_prices.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: visualizations/temporal_stock_prices.png')

### 4b. Categorical/Frequency Analysis: AI Job Market Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

top_jobs = jobs_clean['job_title'].value_counts().head(10)
colors = sns.color_palette('viridis', len(top_jobs))
axes[0].barh(top_jobs.index[::-1], top_jobs.values[::-1], color=colors)
axes[0].set_title('Top 10 AI Job Titles', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Number of Postings', fontsize=11)
for i, v in enumerate(top_jobs.values[::-1]):
    axes[0].text(v + 0.5, i, str(v), va='center', fontsize=9)

if 'experience_label' in jobs_clean.columns:
    exp_order = ['Entry-level', 'Mid-level', 'Senior', 'Executive']
    exp_counts = jobs_clean['experience_label'].value_counts().reindex(exp_order).dropna()
    exp_colors = ['#66BB6A', '#42A5F5', '#FFA726', '#EF5350']
    axes[1].bar(exp_counts.index, exp_counts.values, color=exp_colors[:len(exp_counts)])
    axes[1].set_title('Distribution by Experience Level', fontsize=14, fontweight='bold')
    axes[1].set_ylabel('Number of Jobs', fontsize=11)
    axes[1].tick_params(axis='x', rotation=15)
    for i, v in enumerate(exp_counts.values):
        axes[1].text(i, v + 2, str(v), ha='center', fontsize=10, fontweight='bold')
else:
    exp_col = [c for c in jobs_clean.columns if 'experience' in c.lower()]
    if exp_col:
        jobs_clean[exp_col[0]].value_counts().head(10).plot(kind='bar', ax=axes[1])
        axes[1].set_title('Experience Distribution', fontsize=14, fontweight='bold')

if 'work_setting' in jobs_clean.columns:
    work_counts = jobs_clean['work_setting'].value_counts()
    work_colors = ['#FF7043', '#FFB74D', '#4DB6AC']
    axes[2].pie(
        work_counts.values, labels=work_counts.index, colors=work_colors[:len(work_counts)],
        autopct='%1.1f%%', startangle=90, textprops={'fontsize': 11}
    )
    axes[2].set_title('Remote Work Distribution', fontsize=14, fontweight='bold')
elif 'remote_ratio' in jobs_clean.columns:
    jobs_clean['remote_ratio'].value_counts().plot(kind='bar', ax=axes[2])
    axes[2].set_title('Remote Ratio Distribution', fontsize=14, fontweight='bold')

plt.suptitle('AI Labor Market - Job Distribution Analysis', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(VIZ_DIR, 'categorical_job_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: visualizations/categorical_job_distribution.png')

### 4c. Correlation/Relationship Analysis: Salary Patterns & Stock Correlations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

if 'experience_label' in jobs_clean.columns and 'salary_in_usd' in jobs_clean.columns:
    exp_order = ['Entry-level', 'Mid-level', 'Senior', 'Executive']
    valid_levels = [l for l in exp_order if l in jobs_clean['experience_label'].values]
    sns.boxplot(
        data=jobs_clean, x='experience_label', y='salary_in_usd',
        order=valid_levels, palette='viridis', ax=axes[0]
    )
    axes[0].set_title('Salary by Experience Level', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Experience Level', fontsize=11)
    axes[0].set_ylabel('Salary (USD)', fontsize=11)
    axes[0].tick_params(axis='x', rotation=15)
    axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x:,.0f}'))
elif 'salary_in_usd' in jobs_clean.columns:
    jobs_clean['salary_in_usd'].hist(bins=30, ax=axes[0], color='steelblue')
    axes[0].set_title('Salary Distribution', fontsize=14, fontweight='bold')

if 'work_year' in jobs_clean.columns and 'salary_in_usd' in jobs_clean.columns:
    yearly_salary = jobs_clean.groupby('work_year')['salary_in_usd'].agg(['mean', 'median']).reset_index()
    axes[1].plot(yearly_salary['work_year'], yearly_salary['mean'], 'o-', color='#1976D2',
                 linewidth=2, markersize=8, label='Mean Salary')
    axes[1].plot(yearly_salary['work_year'], yearly_salary['median'], 's--', color='#E64A19',
                 linewidth=2, markersize=8, label='Median Salary')
    axes[1].set_title('AI Salary Trends Over Time', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Year', fontsize=11)
    axes[1].set_ylabel('Salary (USD)', fontsize=11)
    axes[1].legend(fontsize=10)
    axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x:,.0f}'))

stock_clean['Date'] = pd.to_datetime(stock_clean['Date'])
pivot_returns = stock_clean.pivot_table(
    index='Date', columns='Ticker', values='Daily_Return'
).dropna()
if not pivot_returns.empty:
    corr_matrix = pivot_returns.corr()
    sns.heatmap(
        corr_matrix, annot=True, cmap='RdYlBu_r', center=0,
        fmt='.2f', square=True, ax=axes[2],
        cbar_kws={'shrink': 0.8}
    )
    axes[2].set_title('AI Stock Return Correlations', fontsize=14, fontweight='bold')

plt.suptitle('AI Labor Market - Relationship Analysis', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(VIZ_DIR, 'correlation_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: visualizations/correlation_analysis.png')

### 4d. HackerNews Engagement Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].scatter(hn_clean['points'], hn_clean['num_comments'], alpha=0.5,
                c='#1976D2', edgecolors='white', linewidth=0.5, s=50)
axes[0].set_title('HN Story Points vs Comments', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Points', fontsize=11)
axes[0].set_ylabel('Number of Comments', fontsize=11)
axes[0].grid(True, alpha=0.3)

monthly_posts = hn_clean.groupby('year_month').size().reset_index(name='count')
monthly_posts = monthly_posts.sort_values('year_month')
axes[1].fill_between(range(len(monthly_posts)), monthly_posts['count'], alpha=0.3, color='#7B1FA2')
axes[1].plot(range(len(monthly_posts)), monthly_posts['count'], 'o-', color='#7B1FA2', linewidth=2)
step = max(1, len(monthly_posts) // 10)
axes[1].set_xticks(range(0, len(monthly_posts), step))
axes[1].set_xticklabels(monthly_posts['year_month'].iloc[::step], rotation=45, fontsize=8)
axes[1].set_title('Monthly AI Job Discussion Volume on HackerNews', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Number of Stories', fontsize=11)
axes[1].set_xlabel('Month', fontsize=11)

plt.tight_layout()
plt.savefig(os.path.join(VIZ_DIR, 'hn_engagement.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: visualizations/hn_engagement.png')

## 5. Key Insights

### Temporal Trends
- AI/tech stock prices show a strong upward trajectory from 2020-2025, with NVIDIA showing the most dramatic growth.
- Trading volume spikes correlate with major AI announcements.

### Job Market Structure
- The AI job market is heavily skewed toward experienced professionals, with senior-level roles dominating.
- Remote work adoption is notably high in AI roles compared to the broader tech industry.

### Salary Patterns
- Clear positive correlation between experience level and salary.
- Salary growth trend from 2020-2025 reflects increasing demand for AI talent.
- AI company stock returns are positively correlated, meaning these companies tend to move together.

### Cross-Source Connections
- HackerNews discussion volume in AI career topics tracks with broader market interest in AI.
- The upward salary trend aligns with the stock price appreciation, suggesting the overall AI boom is driving both market valuations and compensation growth.